In [16]:
import os, re
import json
from PIL import Image, ImageDraw

# Mapping of image locations to OCR file names
ocr_files = {
    "left_bottom": "0001.json",
    "left_top": "0002.json",
    "right_bottom": "0003.json",
    "right_top": "0004.json"
}
def get_ocr_file_path(image_location, base_folder, year, page):
    # Ensure the image_location is valid
    if image_location not in ocr_files:
        raise ValueError(f"Invalid image location: {image_location}")

    # Get the OCR file name for the given image location
    ocr_file_name = ocr_files[image_location]

    # Construct the full file path for the OCR output
    ocr_file_path = os.path.join(
        base_folder, f'Tokyo_Jobs/Processed_Data/{year}/Page{page:03d}/NDLoutput', ocr_file_name
    )
    return ocr_file_path

def get_page_range(base_folder, year):
    # Construct the path to the year directory
    year_path = os.path.join(base_folder, f'Tokyo_Jobs/Processed_Data/{year}')
    
    # Get a list of all items in the year directory
    try:
        items = os.listdir(year_path)
    except FileNotFoundError:
        print(f"The directory {year_path} does not exist.")
        return None, None

    # Filter out directories with the format "Page###"
    page_dirs = [item for item in items if re.match(r'Page\d+', item)]

    # Extract page numbers from the directory names
    page_numbers = [int(re.search(r'(\d+)', page_dir).group()) for page_dir in page_dirs if re.search(r'(\d+)', page_dir)]

    if page_numbers:
        # Determine the starting and ending page numbers
        start_page = min(page_numbers)
        end_page = max(page_numbers)
        return start_page, end_page
    else:
        print(f"No valid Page### directories found in {year_path}.")
        return None, None

# Function to build text to coordinate mapping
def build_text_to_coord(ocr_data):
    text_to_coord = {}
    for entry in ocr_data:
        if len(entry) == 5:
            x_min, y_min, x_max, y_max, text = entry
            text_to_coord[text] = (x_min, y_min, x_max, y_max)
    return text_to_coord

# Function to group entries into columns based on x-coordinates
def group_into_columns(ocr_data, column_threshold=15):
    text_to_coord = build_text_to_coord(ocr_data)
    columns = {}

    for text, (x_min, y_min, x_max, y_max) in text_to_coord.items():
        column_found = False

        for col_x in columns:
            if abs(col_x - x_min) <= column_threshold:
                columns[col_x]["entries"].append(text)
                columns[col_x]["bbox"] = [
                    [min(columns[col_x]["bbox"][0][0], x_min), min(columns[col_x]["bbox"][0][1], y_min)],
                    [max(columns[col_x]["bbox"][1][0], x_max), max(columns[col_x]["bbox"][1][1], y_max)]
                ]
                column_found = True
                break

        if not column_found:
            columns[x_min] = {"entries": [text], "bbox": [[x_min, y_min], [x_max, y_max]]}

    return columns

# Function to draw bounding boxes on an image
def draw_bounding_boxes_on_image(image_path, columns):
    with Image.open(image_path) as img:
        draw = ImageDraw.Draw(img)

        for col_x, column_data in columns.items():
            bbox = column_data['bbox']
            entries = column_data['entries']
            bbox_flat = (bbox[0][0], bbox[0][1], bbox[1][0], bbox[1][1])
            
            # Draw the rectangle
            draw.rectangle(bbox_flat, outline="red", width=2)

            # Place text at the top-left corner of the bounding box
            for entry in entries:
                if entry in text_to_coord:
                    draw.text((bbox[0][0], bbox[0][1]), entry, fill="blue")

        # Define the save path with the "Annotated" prefix
        annotated_image_path = os.path.join(
            os.path.dirname(image_path), 
            'Annotated' + os.path.basename(image_path)
        )
        
        # Save the annotated image
        img.save(annotated_image_path)
        print(f"Annotated image saved at: {annotated_image_path}")

# Function to save bounding boxes as JSON for later annotation
def save_bounding_boxes_as_json(columns, save_path):
    # Prepare data for JSON serialization
    bbox_data = {}
    for col_x, column_data in columns.items():
        bbox = column_data['bbox']
        entries = column_data['entries']
        bbox_data[f"Column_{col_x}"] = {
            "entries": entries,
            "bounding_box": {
                "top_left": bbox[0],
                "bottom_right": bbox[1]
            }
        }

    # Write the data to a JSON file
    with open(save_path, 'w', encoding='utf-8') as file:
        json.dump(bbox_data, file, indent=4)
    print(f"Bounding box data saved at: {save_path}")
# Read the OCR data from the provided JSON file
User = os.environ.get('USERNAME')  # For Windows
# Define the base folder path
base_folder = f'C:/Users/{User}/Box/Research Notes (keitaro2@illinois.edu)'

Year = 1941
start_page, end_page = get_page_range(base_folder, Year)
error_pages = []

if start_page is not None and end_page is not None:
    print(f"Start Page: {start_page}, End Page: {end_page}")

    # Iterate over the pages and process each one
    for Page in range(start_page, end_page + 1):
        try:
            # Your processing code here
            # For example: draw_bounding_boxes_on_image, save_bounding_boxes_as_json, etc.
            print(f"Processing Page {Page}...")  # Placeholder for actual processing

            for image_location in ocr_files.keys():
                # Get the OCR file path
                file_path = get_ocr_file_path(image_location, base_folder, Year, Page)
                print(f"OCR file path for {image_location}: {file_path}")
                with open(file_path, 'r', encoding='utf-8') as file:                                                                                                                                            
                    ocr_data = json.load(file)[0]                                                             

                image_path = f'C:/Users/{User}/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/{Year}/Page{Page:03d}/img/{image_location}.jpg'
                print(image_path)

                # Generate the text_to_coord mapping
                text_to_coord = build_text_to_coord(ocr_data)

                # Group the entries into columns based on x-coordinates
                columns = group_into_columns(ocr_data)

                # Draw the bounding boxes on the image
                draw_bounding_boxes_on_image(image_path, columns)

                # Define the path to save the JSON file
                json_save_folder = f'C:/Users/{User}/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/{Year}/Page{Page:03d}/Annotation'
                os.makedirs(json_save_folder, exist_ok=True)  # Ensure the folder exists
                json_save_path = os.path.join(json_save_folder, f'{image_location}_ColumnBoundingBoxes.json')


                # Save the bounding boxes as JSON
                save_bounding_boxes_as_json(columns, json_save_path)

            # If an error occurs in the processing code, it will be caught by the except block
            
        except Exception as e:
            print(f"An error occurred while processing Page {Page}: {e}")
            error_pages.append(Page)  # Record the page number that returned an error

    # After all pages have been processed, report any errors
    if error_pages:
        print("Errors occurred on the following pages:", error_pages)
    else:
        print("All pages processed successfully.")

else:
    print("Could not determine the page range.")    

Start Page: 1, End Page: 118
Processing Page 1...
OCR file path for left_bottom: C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)\Tokyo_Jobs/Processed_Data/1941/Page001/NDLoutput\0001.json
C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/1941/Page001/img/left_bottom.jpg
Annotated image saved at: C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/1941/Page001/img\Annotatedleft_bottom.jpg
Bounding box data saved at: C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/1941/Page001/Annotation\left_bottom_ColumnBoundingBoxes.json
OCR file path for left_top: C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)\Tokyo_Jobs/Processed_Data/1941/Page001/NDLoutput\0002.json
C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/1941/Page001/img/left_top.jpg
Annotated image saved at: C:/Users/